# Parsing de Documentos — Parte 5: OCR (Documentos Escaneados)

**Problema:** PDFs escaneados são imagens, não texto.
PyMuPDF, pdfplumber, Docling e Unstructured retornam string vazia.
Precisamos de **OCR** — converter a imagem em texto.

**Ferramentas:**
- `Tesseract` — OCR tradicional, gratuito, offline
- `Gemini (visão)` — LLM multimodal que "lê" imagens
- **Pipeline Tesseract → LLM** — o padrão de produção

In [ ]:
!apt-get install tesseract-ocr tesseract-ocr-por -y -qq
!pip install pytesseract Pillow google-generativeai -q

In [ ]:
import pytesseract
from PIL import Image, ImageDraw, ImageFilter
import random
import numpy as np
from IPython.display import display
import google.generativeai as genai
from google.colab import userdata

In [ ]:
# Configurar Gemini
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
model = genai.GenerativeModel('gemini-2.0-flash')
print("Gemini configurado.")

---
## Bloco 5a — OCR com Tesseract (tradicional)

Tesseract é o motor de OCR open-source mais usado.
Funciona bem em imagens limpas, mas degrada com ruído.

In [ ]:
# Criar imagem simulando nota fiscal escaneada
random.seed(42)
largura, altura = 620, 500
img = Image.new('RGB', (largura, altura), color=(248, 244, 230))
draw = ImageDraw.Draw(img)

draw.rectangle([15, 15, largura-15, altura-15], outline=(160, 140, 120), width=2)

# Nota: usamos apenas ASCII porque a fonte default do Pillow
# nao renderiza acentos corretamente em todos os ambientes
linhas_doc = [
    (40,  40,  "NOTA FISCAL No 00.123 - 2a VIA"),
    (40,  70,  "Emitente: COMERCIO SAO PEDRO LTDA"),
    (40,  95,  "CNPJ: 12.345.678/0001-99    IE: 123.456.789.000"),
    (40, 120,  "End: Rua das Flores, 500 - Centro - Sao Paulo/SP"),
    (40, 155,  "-" * 60),
    (40, 175,  "DESCRICAO                  QTD   UNIT       TOTAL"),
    (40, 195,  "-" * 60),
    (40, 215,  "Cimento CP-II 50kg          10   R$32,00   R$320,00"),
    (40, 235,  "Areia media (saco 20kg)      5   R$18,00    R$90,00"),
    (40, 255,  "Tijolo ceramico (cx 50un)    3   R$95,00   R$285,00"),
    (40, 275,  "Tinta acrilica branca 18L    2  R$120,00   R$240,00"),
    (40, 295,  "-" * 60),
    (40, 320,  "TOTAL GERAL:                             R$935,00"),
    (40, 350,  "Forma de pgto: DINHEIRO    Data: 28/03/2026"),
    (40, 385,  "Assinatura: _____________________________"),
]

for x, y, texto in linhas_doc:
    ox, oy = random.randint(-1, 1), random.randint(-1, 1)
    tom = random.randint(15, 45)
    draw.text((x+ox, y+oy), texto, fill=(tom, tom, tom+5))

# Ruído de scanner
pixels = img.load()
for _ in range(1200):
    rx, ry = random.randint(0, largura-1), random.randint(0, altura-1)
    v = random.randint(130, 210)
    pixels[rx, ry] = (v, v-5, v-10)

img.save('/content/nota_fiscal_scan.png', dpi=(150, 150))
print("Imagem da nota fiscal criada:")
display(img)

In [ ]:
# OCR com Tesseract na imagem limpa
texto_tesseract = pytesseract.image_to_string(img, lang='por')

print("=" * 60)
print("TESSERACT — IMAGEM LIMPA")
print("=" * 60)
print(texto_tesseract)

In [ ]:
# Degradar a imagem: blur + ruído + rotação leve
img_degradada = img.copy()

# Blur
img_degradada = img_degradada.filter(ImageFilter.GaussianBlur(radius=1.5))

# Rotação leve (simula scan torto)
img_degradada = img_degradada.rotate(2, expand=False, fillcolor=(240, 235, 220))

# Mais ruído
pixels_deg = img_degradada.load()
for _ in range(4000):
    rx = random.randint(0, img_degradada.width - 1)
    ry = random.randint(0, img_degradada.height - 1)
    v = random.randint(80, 200)
    pixels_deg[rx, ry] = (v, v, v)

img_degradada.save('/content/nota_fiscal_degradada.png', dpi=(150, 150))
print("Imagem degradada:")
display(img_degradada)

# Tesseract na imagem degradada
texto_degradado = pytesseract.image_to_string(img_degradada, lang='por')

print("\n" + "=" * 60)
print("TESSERACT — IMAGEM DEGRADADA")
print("=" * 60)
print(texto_degradado)
print("\n⚠️  Erros aparecem: caracteres trocados, palavras quebradas, valores errados.")

---
## Bloco 5b — OCR via LLM (Gemini com visão)

A LLM multimodal recebe a **imagem diretamente** e entende o contexto.
Especialmente para tabelas, é **muito melhor** que OCR tradicional.

In [ ]:
# Enviar a MESMA imagem degradada para o Gemini
img_degradada_pil = Image.open('/content/nota_fiscal_degradada.png')

prompt_visao = """Extraia TODO o texto desta imagem de nota fiscal.
Preserve a estrutura da tabela.
Não invente dados — transcreva exatamente o que está visível."""

resposta_llm = model.generate_content([prompt_visao, img_degradada_pil])
texto_llm = resposta_llm.text

print("=" * 60)
print("GEMINI (VISÃO) — MESMA IMAGEM DEGRADADA")
print("=" * 60)
print(texto_llm)

---
## Bloco 5c — Pipeline de produção: Tesseract → LLM

O padrão de produção é um pipeline em dois estágios:
1. Tesseract extrai texto bruto (rápido, barato)
2. LLM recebe **imagem + texto do Tesseract** e corrige

A LLM usa o texto do Tesseract como **âncora** e a imagem como **ground truth**.
Isso reduz alucinação comparado com LLM pura.

In [ ]:
# Pipeline: Tesseract → LLM (correção com âncora)
prompt_correcao = f"""Você recebeu o resultado de um OCR (Tesseract) de uma nota fiscal.
O OCR pode conter erros. Use a IMAGEM como referência para corrigir.

REGRAS:
- Corrija erros de OCR (caracteres trocados, palavras quebradas)
- Preserve a estrutura original (tabela, campos)
- NÃO invente dados que não estão na imagem
- Se não conseguir ler algo, marque como [ilegível]

TEXTO DO OCR (Tesseract):
{texto_degradado}

Corrija o texto acima usando a imagem como referência."""

resposta_pipeline = model.generate_content([prompt_correcao, img_degradada_pil])
texto_pipeline = resposta_pipeline.text

print("=" * 60)
print("PIPELINE: TESSERACT → LLM (CORREÇÃO)")
print("=" * 60)
print(texto_pipeline)

---
## Bloco 5d — Comparação dos três métodos

In [ ]:
# Comparação lado a lado
print("=" * 60)
print("COMPARAÇÃO DOS 3 MÉTODOS (mesma imagem degradada)")
print("=" * 60)

print("\n" + "-" * 40)
print("1. TESSERACT PURO")
print("-" * 40)
print(texto_degradado[:500])

print("\n" + "-" * 40)
print("2. LLM PURA (Gemini visão)")
print("-" * 40)
print(texto_llm[:500])

print("\n" + "-" * 40)
print("3. PIPELINE: TESSERACT → LLM")
print("-" * 40)
print(texto_pipeline[:500])

print("\n" + "=" * 60)
print("ANÁLISE")
print("=" * 60)
print("""
| Método           | Custo    | Latência | Precisão | Tabelas |
|------------------|----------|----------|----------|---------|
| Tesseract puro   | Gratuito | ~ms      | Baixa*   | Ruim    |
| LLM pura (visão) | $$       | ~seg     | Alta     | Ótima   |
| Tesseract + LLM  | $        | ~seg     | Muito alta| Ótima  |

* Em imagens degradadas. Em imagens limpas, Tesseract é razoável.

→ O pipeline Tesseract → LLM é o padrão em produção:
  OCR dá a base, LLM refina com contexto visual.
""")

---
## Trade-offs

| Método | ✅ Vantagens | ❌ Limitações |
|---|---|---|
| **Tesseract** | Gratuito, offline, rápido | Erros em imagens ruins, não preserva layout |
| **LLM (visão)** | Entende contexto, ótimo para tabelas | Custo por token, pode alucinar |
| **Tesseract + LLM** | Melhor dos dois mundos, menos alucinação | Dois estágios, custo da LLM |

> O pipeline **Tesseract → LLM** é o padrão de produção real:
> o OCR tradicional dá a base, a LLM refina.